In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES']='0'
import pandas as pd 
import anndata 
import numpy as np 
import warnings
import time
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", FutureWarning)
import pickle as pkl
import os.path as osp
import scanpy as sc
import squidpy as sq
%config InlineBackend.print_figure_kwargs={'facecolor' : "w"}
%config InlineBackend.figure_format='retina'

/home/users/yuling/anaconda3/envs/my_mos/lib/python3.12/site-packages/numba/core/decorators.py:246: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


In [2]:
import moscot as mt
from moscot import datasets
from moscot.problems.space import AlignmentProblem
from sklearn.metrics import adjusted_rand_score

DLPFC data

In [ ]:
# load data and normalize
# ['151669', '151670',  '151671', '151672']
# ['151673', '151674',  '151675', '151676']
slice_ids = ['151507', '151508',  '151509', '151510']
slices=[]
for j,section_ids in enumerate(slice_ids):
    slice1 = load_DLPFC(section_id=section_ids)
    slice1.layers["counts"] = slice1.X.copy()
    print(slice1.X)
    sc.pp.normalize_total(slice1)
    sc.pp.log1p(slice1)
    slice1.obs['Z']=str(j)
    slice1.obs['slice_id']=section_ids
    slices.append(slice1)
adata=anndata.concat(slices)
# prepare the alignment problem
ap = AlignmentProblem(adata=adata)
ap = ap.prepare(batch_key="Z", policy="sequential")
ap = ap.solve(alpha=0.8)
solutions = ap.solutions  
# Access the computed solutions
for cur_idx in range(1,len(slice_ids)):
    index=str(slice_ids[cur_idx-1])+'_'+str(slice_ids[cur_idx])
    map_path=osp.join('/home/data3/yuling/xenium_region2_5599/moscot/DLPFC_4/151507_151510', index+'_AlignmentPi.pkl')
    with open(map_path, "wb") as f:
        pkl.dump(solutions[tuple([str(cur_idx-1), str(cur_idx)])].transport_matrix, f)
# get aligned spatial coordinates
ap.align(reference="0", mode="affine", key_added="spatial_affine")
 
adata.obsm['spatial_3D']=np.concatenate([adata.obsm['spatial_affine'],np.array(adata.obs['Z'].values)[:,None]],axis=1)
df = pd.DataFrame(adata.obsm['spatial_3D'],
                  columns=['X', 'Y','Z'],
                  index=adata.obs_names.to_list())
df['original_clusters']=adata.obs['original_clusters'].values
df['slice']=adata.obs['slice_id'].values
df.to_csv(osp.join('/home/data3/yuling/xenium_region2_5599/moscot/DLPFC_4/151507_151510','3d_coord_151507.csv')) 

Merfish data

In [ ]:
in_ = '/home/data3/yuling/xenium_region2_5599/merfish_mouse_brain'
slice_ids_o = range(33)
sliceID =[[str(x) for x in slice_ids_o]]
end_no = 33
slices = [] #this is the original data
slice_ids = sliceID[0]
for i in range(len(slice_ids)):
    slice1 = sc.read(osp.join(in_, 'merfish_mouse_brain_slice'+slice_ids[i]+'.h5ad'))
    slice1.obs['original_clusters']=slice1.obs['spa_cluster'].astype('category')
    slice1.obsm['spatial'] = slice1.obsm['spatial'].to_numpy()[:, :2]
    slice1.obs['Z']=str(i)
    slices.append(slice1)
adata=anndata.concat(slices)
ap = AlignmentProblem(adata=adata)
ap = ap.prepare(batch_key="Z", policy="sequential")
ap = ap.solve(alpha=0.8)
solutions = ap.solutions  
# Access the computed solutions
for cur_idx in range(1,len(slice_ids)):
    index=str(slice_ids[cur_idx-1])+'_'+str(slice_ids[cur_idx])
    map_path=osp.join('/home/data3/yuling/xenium_region2_5599/moscot/Merfish_mouse', index+'_AlignmentPi.pkl')
    with open(map_path, "wb") as f:
        pkl.dump(solutions[tuple([str(cur_idx-1), str(cur_idx)])].transport_matrix, f)

ap.align(reference="0", mode="affine", key_added="spatial_affine")

adata.obsm['spatial_3D']=np.concatenate([adata.obsm['spatial_affine'],np.array(adata.obs['Z'].values)[:,None]],axis=1)
df = pd.DataFrame(adata.obsm['spatial_3D'],
                columns=['X', 'Y','Z'],
                index=adata.obs_names.to_list())
df['original_clusters']=adata.obs['original_clusters'].values
df['slice']=adata.obs['slice_id'].values
df.to_csv(osp.join('/home/data3/yuling/xenium_region2_5599/moscot/Merfish_mouse','3d_coord_merfish.csv')) 

mHypothalamus data

In [ ]:
slices=[]
slice_ids = ['-0.04', '-0.09',  '-0.14', '-0.19' , '-0.24']
for j,section_ids in enumerate(slice_ids):
    slice1 = load_mHypothalamus(section_id=section_ids)
    slice1.var_names_make_unique()
    slice1.layers["counts"] = slice1.X.copy()
    sc.pp.normalize_total(slice1)
    sc.pp.log1p(slice1)
    slice1.obs['Z']=str(j)
    slice1.obs["Z"]=slice1.obs["Z"].astype("category")
    slice1.obs['slice_id']=section_ids
    slices.append(slice1)
adata=anndata.concat(slices)
# prepare the alignment problem
ap = AlignmentProblem(adata=adata)
ap = ap.prepare(batch_key="Z", policy="sequential")
# get alignment matrix
solutions = ap.solutions  
# Access the computed solutions
for cur_idx in range(1,len(slice_ids)):
    index=str(slice_ids[cur_idx-1])+'_'+str(slice_ids[cur_idx])
    map_path=osp.join('/home/data3/yuling/xenium_region2_5599/moscot/mHy', index+'_AlignmentPi.pkl')
    with open(map_path, "wb") as f:
        pkl.dump(solutions[tuple([str(cur_idx-1), str(cur_idx)])].transport_matrix, f)
 
ap.align(reference="0", mode="affine", key_added="spatial_affine")

adata.obsm['spatial_3D']=np.concatenate([adata.obsm['spatial_affine'],np.array(adata.obs['Z'].values)[:,None]],axis=1)
df = pd.DataFrame(adata.obsm['spatial_3D'],
                  columns=['X', 'Y','Z'],
                  index=adata.obs_names.to_list())
df['original_clusters']=adata.obs['original_clusters'].values
df['slice']=adata.obs['slice_id'].values
df.to_csv(osp.join('/home/data3/yuling/xenium_region2_5599/moscot/mHy','3d_coord_mHy.csv')) 

simulated data

In [ ]:
slice_ids =[151673,100,80,60,40,20]
slices=[]
slice1=sc.read_h5ad('/home/data3/yuling/xenium_region2_5599/simData/DLPFC_151673original_spotIndex.h5')
slice1.layers["counts"] = slice1.X.copy()
sc.pp.normalize_total(slice1)
sc.pp.log1p(slice1)
slice1.obs['slice_id']=151673
slice1.obs['Z']="0"
slice1.obs_names_make_unique()
slices.append(slice1)
for index,s in enumerate(slice_ids[1:]):
    slice1=sc.read_h5ad(osp.join('/home/data3/yuling/xenium_region2_5599/simData/DLPFC_151673_overlap='+str(s)+'__pseudocount_0_spotIndex.h5'))
    slice1.layers["counts"] = slice1.X.copy()
    sc.pp.normalize_total(slice1)
    sc.pp.log1p(slice1)
    slice1.obs['slice_id']=s
    slice1.obs['Z']=str(index+1)
        
    slice1.obs_names_make_unique()
    slices.append(slice1) 
adata=anndata.concat(slices)
ap = AlignmentProblem(adata=adata)
ap = ap.prepare(batch_key="Z", policy="sequential")
ap = ap.solve(alpha=0.9)
solutions = ap.solutions  
# Access the computed solutions
for cur_idx in range(1,len(slice_ids)):
    index=str(slice_ids[cur_idx-1])+'_'+str(slice_ids[cur_idx])
    map_path=osp.join('/home/data3/yuling/xenium_region2_5599/moscot/simu_data', index+'_AlignmentPi.pkl')
    with open(map_path, "wb") as f:
        pkl.dump(solutions[tuple([str(cur_idx-1), str(cur_idx)])].transport_matrix, f)

ap.align(reference="0", mode="affine", key_added="spatial_affine")

adata.obsm['spatial_3D']=np.concatenate([adata.obsm['spatial_affine'],np.array(adata.obs['Z'].values)[:,None]],axis=1)
df = pd.DataFrame(adata.obsm['spatial_3D'],
                  columns=['X', 'Y','Z'],
                  index=adata.obs_names.to_list())
df['original_clusters']=adata.obs['original_clusters'].values
df['slice']=adata.obs['slice_id'].values
df.to_csv(osp.join('/home/data3/yuling/xenium_region2_5599/moscot/simu_data','3d_coord_simu.csv')) 


4 slices of MOSTA embryo

In [ ]:
slices=[]
slice_ids = ['E11.5_E1S1.MOSTA', 'E11.5_E1S2.MOSTA',  'E11.5_E1S3.MOSTA', 'E11.5_E1S4.MOSTA']
# load data and normalize
for j,section_ids in enumerate(slice_ids):
    slice1 = load_Mostaembryo(section_id=section_ids)
    sc.pp.normalize_total(slice1)
    sc.pp.log1p(slice1)
    slice1.obs['Z']=str(j)
    slice1.obs['slice_id']=section_ids
    slices.append(slice1)
adata=anndata.concat(slices)
# prepare the alignment problem
ap = AlignmentProblem(adata=adata)
ap = ap.prepare(batch_key="Z", policy="sequential")
ap = ap.solve(alpha=0.8)
solutions = ap.solutions  
# Access the computed solutions
for cur_idx in range(1,len(slice_ids)):
    index=str(slice_ids[cur_idx-1])+'_'+str(slice_ids[cur_idx])
    map_path=osp.join('/home/data3/yuling/xenium_region2_5599/moscot/mosta_embryo', index+'_AlignmentPi.pkl')
    with open(map_path, "wb") as f:
        pkl.dump(solutions[tuple([str(cur_idx-1), str(cur_idx)])].transport_matrix, f)

ap.align(reference="0", mode="affine", key_added="spatial_affine")

adata.obsm['spatial_3D']=np.concatenate([adata.obsm['spatial_affine'],np.array(adata.obs['Z'].values)[:,None]],axis=1)
df = pd.DataFrame(adata.obsm['spatial_3D'],
                columns=['X', 'Y','Z'],
                index=adata.obs_names.to_list())
df['original_clusters']=adata.obs['original_clusters'].values
df['slice']=adata.obs['slice_id'].values
df.to_csv(osp.join('/home/data3/yuling/xenium_region2_5599/moscot/mosta_embryo','3d_coord_mosta.csv')) 